# Watching Differential Evolution in Real Time

This notebook shows how to run **Differential Evolution (DE)** on the classic
**Rastrigin function** and watch the population move across the landscape
in real time, using `metaheuristic‑designer` and Plotly.

You will learn how to:
* define a custom benchmark with a throttle (so the evolution is visible)
* assemble a DE strategy
* run the algorithm manually and update a live contour plot

In [ ]:
import time
import numpy as np
import metaheuristic_designer as mhd
from metaheuristic_designer.benchmarks import Rastrigin
import plotly.graph_objects as go

## 1. Create a throttled Rastrigin function

The standard Rastrigin is too fast for a real‑time display.
We wrap it so each evaluation is delayed by 0.05 seconds, giving us
time to see the population move.

In [ ]:
class ThrottledRastrigin(Rastrigin):
    def __init__(self, dimension=2, mode="min", sleep=0.05):
        super().__init__(dimension, mode=mode)
        self.sleep = sleep

    def objective(self, solution):
        time.sleep(self.sleep)  # slow down for visualisation
        return super().objective(solution)


objfunc = ThrottledRastrigin(dimension=2, mode="min", sleep=0.05)

## 2. Set the random seed

For reproducibility we fix the random generator.

In [ ]:
rng = mhd.check_random_state(42)

## 3. Build the DE strategy

We use a population of 30 individuals, the `DE/rand/1` mutation,
and a uniform initialisation between the function’s bounds.

In [ ]:
strategy = mhd.strategies.DE(
    initializer=mhd.initializers.UniformInitializer(objfunc.dimension, objfunc.lower_bound, objfunc.upper_bound, pop_size=30, random_state=rng),
    de_operator_name="DE/rand/1",
    F=0.5,
    Cr=0.7,
    random_state=rng,
)

## 4. Draw the background contour

We evaluate the **original** (fast) Rastrigin on a fine grid
to obtain a contour map that will stay static during the animation.

In [ ]:
orig_rastrigin = Rastrigin(dimension=2, mode="min")

n_grid = 200
x_vals = np.linspace(objfunc.lower_bound, objfunc.upper_bound, n_grid)
y_vals = np.linspace(objfunc.lower_bound, objfunc.upper_bound, n_grid)
X, Y = np.meshgrid(x_vals, y_vals)

grid_points = np.c_[X.ravel(), Y.ravel()]
Z = np.array([orig_rastrigin.objective(pt) for pt in grid_points]).reshape(X.shape)

## 5. Create the algorithm and initialise the population

We ask the algorithm to record the whole population at each generation
(`track_full_population=True`) so we can access the positions later.
The `tqdm` reporter shows a progress bar.

In [ ]:
algo = mhd.Algorithm(
    objfunc,
    strategy,
    stop_cond="max_iterations",
    max_iterations=80,
    reporter="tqdm",
    track_full_population=True,
)

population = algo.initialize()
algo.stopping_condition.restart()
algo.stopping_condition.step(population)

## 6. Set up the live Plotly figure

We create a `FigureWidget` with a contour of the Rastrigin landscape,
a scatter trace for the population (red dots), and a star for the best
individual found so far.

In [ ]:
fig = go.FigureWidget()
fig.add_trace(
    go.Contour(
        x=x_vals,
        y=y_vals,
        z=Z,
        colorscale="viridis",
        showscale=False,
        contours=dict(start=0, end=100, size=4),
    )
)
fig.add_trace(go.Scatter(x=[], y=[], mode="markers", marker=dict(color="red", size=6), name="population"))
fig.add_trace(go.Scatter(x=[], y=[], mode="markers", marker=dict(color="yellow", size=10, symbol="star"), name="best"))
fig.update_layout(
    title="DE on Rastrigin (real-time)",
    xaxis_title="x1",
    yaxis_title="x2",
    xaxis_range=[objfunc.lower_bound, objfunc.upper_bound],
    yaxis_range=[objfunc.lower_bound, objfunc.upper_bound],
    width=700,
    height=600,
)

print("plot setup done")

## 7. Run the optimisation loop

We step through the generations manually, updating the history tracker
and the Plotly figure after each generation. The plot should start
animating immediately.

In [ ]:
# Ensure the plot shows in this cell
display(fig)

algo.restart()
algo.reporter.log_init(algo)
while not algo.stopping_condition.is_finished():
    population = algo.step(population=population)
    algo.history_tracker.step(algo)
    algo.stopping_condition.step(algo.population)
    algo.reporter.log_step(algo)

    # Extract data from the history tracker
    positions = algo.history_tracker.complete_population[-1]  # (pop_size, 2)
    best_sol = algo.history_tracker.best_solutions[-1]  # (2,)

    with fig.batch_update():
        fig.data[1].x = positions[:, 0]
        fig.data[1].y = positions[:, 1]
        fig.data[2].x = [best_sol[0]]
        fig.data[2].y = [best_sol[1]]

algo.reporter.log_end(algo)